# AI labelled data generation and Reward Modelling

In [1]:
import random
import uuid
import json
import os
import sys
from pathlib import Path

path = Path.cwd().parent.absolute()
nyx_path = f'{path}/'
print(nyx_path)
sys.path.append(nyx_path)
from nyx.data_generation import Controller
from nyx.data_generation.settings import BASELINE_LEE_ET_AL
from nyx.data_loaders import HumanEvaluatedDataLoader
from nyx.constants import METRICS_PATH, COMMON_OUTPUT_PATHS

/Users/owner/PycharmProjects/demerzel/


/Users/owner/opt/anaconda3/envs/llm-diss/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


In [2]:
RANDOM_SEED = 42
# PRECISION = torch.float32
PRECISION_NAME = 'float16'
DEVICE = "mps"
LABELLER_MODEL = "bigscience/mt0-small"  # "google/flan-t5-small"
# "stabilityai/stablelm-2-zephyr-1_6b"
# "microsoft/phi-1_5"
# "microsoft/Phi-3-mini-4k-instruct"
# "google/flan-t5-large"
# "google/flan-t5-xl"
# "bigscience/mt0-small"
# "bigscience/mt0-large"
# "bigscience/mt0-xl"
GEMMA_PATH = "/Users/owner/Documents/AI_MSc/13.Dissertation/experiments/labelling-model/gemma-2b-it/"
RUN_ID = "test"  #'316a2787976e4e848ea635422e2ef684' # uuid.uuid4().hex
TESTING = True

In [4]:
config = {
    'llm_model_name': LABELLER_MODEL,  # LABELLER_MODEL, # GEMMA_PATH, # 70B param model
    'precision_name': PRECISION_NAME,
    'device': DEVICE,
    # 'dataset': data,
    'run_id': RUN_ID,
    'max_new_tokens': 512,
}


data_generator = Controller(
    labelling_method=f'{BASELINE_LEE_ET_AL}_single_gpu',  # BASELINE_LEE_ET_AL, # Toth et al, (Ablation)
    labelling_config=config,
    data_loader=HumanEvaluatedDataLoader,
)

if TESTING is True:
    indices = random.sample(range(0, 92859), 4)
    # print(indices)
    data_generator.data_to_label["train"] = data_generator.data_to_label[
        "train"
    ].select(indices)
    data_generator.data_to_label["validation"] = data_generator.data_to_label[
        "validation"
    ].select(range(50))


# print(data_generator.data_to_label)
# data_generator.label_data()
# data_generator.report_on_performance()
# (data_generator.data_to_label['train'])

In [5]:
data_generator.data_to_label

DatasetDict({
    train: Dataset({
        features: ['subreddit', 'post', 'choice', 'candidate_summary_1', 'candidate_summary_2'],
        num_rows: 4
    })
    validation: Dataset({
        features: ['subreddit', 'post', 'choice', 'candidate_summary_1', 'candidate_summary_2'],
        num_rows: 50
    })
})

In [27]:
# from typing import List
# from datasets import DatasetDict
# from pprint import pprint
# def dataset_dict_to_langchain_batch_consumable(data: DatasetDict,
#                                                requested_cols: List[str],
#                                                data_split: str = 'train', ) -> List[dict]:
#     requested_data = data[data_split]
#     data_for_langchain = []
#     for values in zip(*[requested_data[col] for col in requested_cols]):
#         # print(values)
#         row_value = {col: values[index] for index, col in enumerate(requested_cols)}
#         data_for_langchain.append(row_value)

#     return data_for_langchain
# pprint(dataset_dict_to_langchain_batch_consumable(data_generator.data_to_label, ['post', 'candidate_summary_1', 'candidate_summary_2']))

In [ ]:
COMMON_OUTPUT_PATHS = COMMON_OUTPUT_PATHS.format(RUN_ID=RUN_ID)
METRICS_PATH = METRICS_PATH.format(COMMON_OUTPUT_PATHS=COMMON_OUTPUT_PATHS)

if not os.path.exists(METRICS_PATH):
    os.makedirs(METRICS_PATH)

data_path = f'{METRICS_PATH}/data-generation-info.json'

results_dict = {
    'run_id': RUN_ID,
    'labeller_model': LABELLER_MODEL,
    'precision': PRECISION_NAME,
}
with open(data_path, 'w') as file:
    json.dump(results_dict, file)

print('Data generation done and the configuration info is saved.')
print(data_path)

# END